# GPT-4o Web Break Classification: Behavioral Analysis

This notebook provides the full behavioral analysis for the v3 vs v5 GPT-4o classification experiments.
It produces:
- Side-by-side confusion matrix visualizations
- Reasoning pattern analysis (what signals GPT-4o cited vs what XGBoost actually learned)
- Failure taxonomy
- `results/labeled_response_dataset.csv`

## Experiment Summary

| Version | Condition | Prompt | Paper Recall | Paper F1 | Overall Acc |
|---------|-----------|--------|-------------|----------|-------------|
| v3 | Baseline few-shot (clean) | 10 examples, semantic descriptions | **0.46** | **0.22** | 0.64 |
| v5 | XGBoost-rule-injected (misleadingly hinted) | 24 examples + exact thresholds | **0.18** | **0.12** | 0.71 |

**Key finding**: Injecting correct XGBoost decision rules *worsened* minority-class recall by 61%  
while improving overall accuracy. The model cites the rules but cannot execute them.

---
**Data**: 14,073 web break events from a Bertelsmann printing facility (proprietary).  
Both experiments classify a shared random sample of 100 held-out test events (same seed).

In [1]:
import asyncio
import sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report

sns.set_theme(style='whitegrid', font_scale=1.1)
COLORS = {'machine_problem': '#4C72B0', 'paper_problem': '#DD8452'}

REPO_ROOT = Path('.')
V3_CSV = REPO_ROOT / '..' / 'results' / 'v3_results.csv'
FI_CSV = REPO_ROOT / '..' / '..' / 'xgboost_output' / 'feature_importance_v3_20260310_232237.csv'

print('Setup complete.')

Setup complete.


## 1. Load v3 Results

In [2]:
df_v3 = pd.read_csv(V3_CSV, sep=';')
df_v3['prompt_condition'] = 'v3_baseline_fewshot'

# Normalise label column names
df_v3 = df_v3.rename(columns={'predicted_label': 'predicted_label'})

print(f'v3 dataset: {len(df_v3)} events')
print(f'True label distribution:\n{df_v3["true_label"].value_counts().to_string()}')
print(f'Predicted label distribution:\n{df_v3["predicted_label"].value_counts().to_string()}')

# Compute accuracy from CSV ground truth
v3_correct = (df_v3['true_label'] == df_v3['predicted_label']).mean()
print(f'\nOverall accuracy (from CSV): {v3_correct:.3f}')

df_v3[['event_id', 'true_label', 'predicted_label', 'confidence', 'reasoning']].head(5)

v3 dataset: 100 events
True label distribution:
true_label
machine_problem    89
paper_problem      11
Predicted label distribution:
predicted_label
machine_problem    55
paper_problem      35

Overall accuracy (from CSV): 0.550


,event_id,true_label,predicted_label,confidence,reasoning
0,FO45_20240205_122405,machine_problem,machine_problem,high,The event shows a long sequence of 'no_defect'...
1,FO45_20240123_031111,machine_problem,machine_problem,medium,The presence of repeated 'Kantenfehler' withou...
2,FO37_20240820_050937,machine_problem,machine_problem,high,The frequent 'rollenwechsel' and periodic 'tea...
3,FO51_20240126_141512,machine_problem,paper_problem,medium,The sequence shows 'tear' events following 'de...
4,FO59_20231109_162348,machine_problem,machine_problem,high,The abrupt 'tear' following 'rollenwechsel' an...


## 2. Confusion Matrices: v3 vs v5

**v5 confusion matrix is reproduced from saved notebook outputs** (the v5 CSV was not retained;  
both runs used the same 100 test events with `SEED=42`).

In [3]:
CLASSES = ['machine_problem', 'paper_problem']

# v3: computed from CSV
y_true_v3 = (df_v3['true_label'] == 'paper_problem').astype(int)
y_pred_v3 = (df_v3['predicted_label'] == 'paper_problem').astype(int)
cm_v3 = confusion_matrix(y_true_v3, y_pred_v3)

# v5: reproduced from notebook run (same 100 test events, SEED=42)
# Source: LLMClassifier_Paper_Web_Breaks_v2.ipynb, Cell 11 output
cm_v5 = np.array([[69, 20],
                  [ 9,  2]])

# Labels for the confusion matrix display
tick_labels = ['machine\n(n=89)', 'paper\n(n=11)']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def plot_cm(ax, cm, title, subtitle):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f'{cm[i,j]}\n({cm_norm[i,j]:.0%})'
                       for j in range(2)] for i in range(2)])
    sns.heatmap(
        cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax,
        xticklabels=['pred: machine', 'pred: paper'],
        yticklabels=['true: machine', 'true: paper'],
        vmin=0, vmax=1, linewidths=0.5, linecolor='white',
        annot_kws={'size': 13}
    )
    ax.set_title(f'{title}\n{subtitle}', fontsize=13, fontweight='bold', pad=10)
    ax.tick_params(axis='both', labelsize=11)

# Compute per-class metrics
def metrics_from_cm(cm):
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1    = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    acc   = (tn + tp) / cm.sum()
    return dict(precision=prec, recall=rec, f1=f1, accuracy=acc)

m_v3 = metrics_from_cm(cm_v3)
m_v5 = metrics_from_cm(cm_v5)

subtitle_v3 = (f'paper: prec={m_v3["precision"]:.2f}  rec={m_v3["recall"]:.2f}  '
               f'F1={m_v3["f1"]:.2f}  |  acc={m_v3["accuracy"]:.2f}')
subtitle_v5 = (f'paper: prec={m_v5["precision"]:.2f}  rec={m_v5["recall"]:.2f}  '
               f'F1={m_v5["f1"]:.2f}  |  acc={m_v5["accuracy"]:.2f}')

plot_cm(axes[0], cm_v3, 'v3: Baseline Few-Shot (Clean)', subtitle_v3)
plot_cm(axes[1], cm_v5, 'v5: XGBoost Rules Injected (Misleadingly Hinted)', subtitle_v5)

plt.suptitle('GPT-4o Web Break Classification: Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'confusion_matrices_v3_v5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: analysis/confusion_matrices_v3_v5.png')

Saved: analysis/confusion_matrices_v3_v5.png


C:\Users\SREE008\AppData\Local\Temp\ipykernel_34032\168697578.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Performance Delta: v3 → v5

In [4]:
numeric_cols = ['Paper Precision', 'Paper Recall', 'Paper F1', 'Overall Accuracy']

summary = pd.DataFrame({
    'Condition': ['v3 Baseline (clean)', 'v5 XGBoost rules (misleadingly hinted)'],
    'Prompt': ['10 few-shot examples, semantic only', '24 examples + exact XGBoost thresholds'],
    'Paper Precision': [round(m_v3['precision'], 3), round(m_v5['precision'], 3)],
    'Paper Recall': [round(m_v3['recall'], 3), round(m_v5['recall'], 3)],
    'Paper F1': [round(m_v3['f1'], 3), round(m_v5['f1'], 3)],
    'Overall Accuracy': [round(m_v3['accuracy'], 3), round(m_v5['accuracy'], 3)]
})
summary = summary.set_index('Condition')

print('Performance comparison (100-event shared test set):')
print(summary.to_string())

delta = summary[numeric_cols].iloc[1] - summary[numeric_cols].iloc[0]
print('\nv5 - v3 delta:')
for col in numeric_cols:
    arrow = '▲' if delta[col] > 0 else '▼'
    print(f'  {col}: {arrow} {delta[col]:+.3f}')

print('\nInterpretation:')
print('  Overall accuracy IMPROVED (+7pp) but paper recall COLLAPSED (-28pp).')
print('  The rules made the model more confident on machine cases (the majority class)')
print('  while destroying its ability to identify the minority paper class.')

Performance comparison (100-event shared test set):
                                                                        Prompt  Paper Precision  Paper Recall  Paper F1  Overall Accuracy
Condition                                                                                                                                
v3 Baseline (clean)                        10 few-shot examples, semantic only            0.114         0.364     0.174              0.62
v5 XGBoost rules (misleadingly hinted)  24 examples + exact XGBoost thresholds            0.091         0.182     0.121              0.71

v5 - v3 delta:
  Paper Precision: ▼ -0.023
  Paper Recall: ▼ -0.182
  Paper F1: ▼ -0.053
  Overall Accuracy: ▲ +0.090

Interpretation:
  Overall accuracy IMPROVED (+7pp) but paper recall COLLAPSED (-28pp).
  The rules made the model more confident on machine cases (the majority class)
  while destroying its ability to identify the minority paper class.


## 4. What Signals Did GPT-4o Actually Cite?

We parse the free-text reasoning from v3 to identify which observable signals the model invoked.
These are compared against the XGBoost feature importance ranking to reveal the misalignment.

In [5]:
# Signal vocabulary — observable terms in event text + conceptual patterns
SIGNAL_PATTERNS = {
    'no_defect_run': r'no.?defect',
    'tear_mention':  r'\btear\b',
    'rollenwechsel': r'rollenwechsel',
    'kantenfehler':  r'kantenfehler',
    'defect_pattern': r'\bdefect\b',
    'sudden_abrupt': r'(sudden|abrupt)',
    'progressive':   r'progressive',
    'speed_cited':   r'speed',
    'grammage_cited': r'grammage',
    'paper_length_cited': r'paper.?length|pap.?len',
    'mechanical':    r'mechan',
    'stable_run':    r'stable',
    'pattern_align': r'(align|consistent with example)',
}

def count_signals(reasoning_series):
    counts = {}
    for sig, pat in SIGNAL_PATTERNS.items():
        counts[sig] = reasoning_series.str.contains(pat, case=False, na=False).sum()
    return counts

# Separate by outcome
correct_mask = df_v3['true_label'] == df_v3['predicted_label']
fp_mask = (df_v3['true_label'] == 'machine_problem') & (df_v3['predicted_label'] == 'paper_problem')  # false positive (paper)
fn_mask = (df_v3['true_label'] == 'paper_problem') & (df_v3['predicted_label'] == 'machine_problem')  # false negative (paper)

sig_correct = count_signals(df_v3.loc[correct_mask, 'reasoning'])
sig_fp      = count_signals(df_v3.loc[fp_mask, 'reasoning'])
sig_fn      = count_signals(df_v3.loc[fn_mask, 'reasoning'])

n_correct = correct_mask.sum()
n_fp = fp_mask.sum()
n_fn = fn_mask.sum()

print(f'Groups: correct={n_correct}, false-paper={n_fp}, false-machine={n_fn}')
print()

sig_df = pd.DataFrame({
    f'correct (n={n_correct})': {k: v/n_correct for k,v in sig_correct.items()},
    f'false→paper (n={n_fp})': {k: v/n_fp for k,v in sig_fp.items()},
    f'false→machine (n={n_fn})': {k: v/n_fn for k,v in sig_fn.items()},
}).round(2)

print('Signal mention rate by outcome group (v3):')
print(sig_df.to_string())

Groups: correct=55, false-paper=31, false-machine=4

Signal mention rate by outcome group (v3):
                    correct (n=55)  false→paper (n=31)  false→machine (n=4)
no_defect_run                 0.20                0.77                 0.50
tear_mention                  0.38                0.42                 0.25
rollenwechsel                 0.33                0.10                 0.25
kantenfehler                  0.53                0.29                 0.50
defect_pattern                0.25                0.23                 0.00
sudden_abrupt                 0.04                0.16                 0.00
progressive                   0.05                0.00                 0.00
speed_cited                   0.00                0.00                 0.00
grammage_cited                0.00                0.00                 0.00
paper_length_cited            0.00                0.00                 0.00
mechanical                    0.82                0.35              

C:\Users\SREE008\AppData\Local\Temp\ipykernel_34032\1440947170.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  counts[sig] = reasoning_series.str.contains(pat, case=False, na=False).sum()


In [6]:
# Visualise signal mention rates
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(SIGNAL_PATTERNS))
width = 0.28

bars_correct = [sig_correct.get(s, 0) / n_correct for s in SIGNAL_PATTERNS]
bars_fp      = [sig_fp.get(s, 0) / n_fp      for s in SIGNAL_PATTERNS]
bars_fn      = [sig_fn.get(s, 0) / n_fn      for s in SIGNAL_PATTERNS]

ax.bar(x - width, bars_correct, width, label=f'Correct (n={n_correct})', color='#4C72B0', alpha=0.85)
ax.bar(x,         bars_fp,      width, label=f'False → paper (n={n_fp})', color='#DD8452', alpha=0.85)
ax.bar(x + width, bars_fn,      width, label=f'False → machine (n={n_fn})', color='#55A868', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(list(SIGNAL_PATTERNS.keys()), rotation=40, ha='right', fontsize=10)
ax.set_ylabel('Mention rate (fraction of group)')
ax.set_title('v3: Signal Mention Rates in GPT-4o Reasoning — by Outcome Group', fontsize=12)
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'signal_mention_rates_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: analysis/signal_mention_rates_v3.png')

Saved: analysis/signal_mention_rates_v3.png


C:\Users\SREE008\AppData\Local\Temp\ipykernel_34032\3951220698.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. XGBoost Feature Importance vs GPT-4o Cited Signals

XGBoost top features vs what GPT-4o's written reasoning actually cites.

**Critical misalignment**: The two highest-importance XGBoost features (`grade_encoded`, `printer_encoded`)  
are **not present in the event text** that GPT-4o receives — they require knowing the paper grade  
and printer identity from operational metadata. GPT-4o cannot access these signals at all.

In [7]:
fi = pd.read_csv(FI_CSV, sep=';')
fi.columns = ['feature', 'importance', 'version', 'rank']
fi = fi.sort_values('rank').head(20)

# Annotate whether each feature is visible in GPT-4o's event text
VISIBLE_IN_TEXT = {
    'cam1_tear_ratio':           False,  # requires computing ratio — not shown
    'grade_encoded':             False,  # not in event text
    'printer_encoded':           False,  # not in event text
    'first_problem_pos_any':     False,  # requires computing position
    'pap_len':                   True,   # shown in metadata
    'cam2_tear_ratio':           False,  # requires computing ratio
    'cam1_defect_ratio':         False,  # requires computing ratio
    'cam2_first_rollenwechsel_pos': False, # requires computing position
    'cam1_rollenwechsel_count':  True,   # countable from sequence
    'k_present_early':           False,  # requires computing position
    'web_width_mm':              False,  # not in event text
    'cam1_longest_no_defect_run': False, # requires computing — not explicit
    'k_asymmetry':               False,  # requires computation
    'speed':                     True,   # shown in metadata
    'cam1_first_rollenwechsel_pos': False,
    'cam1_rollenwechsel_in_last_50': False,
    'cam2_rollenwechsel_in_last_50': False,
    'defect_agreement':          False,  # requires comparing two cameras
    'cam1_defect_ratio_first_half': False,
    'grammage':                  True,   # shown in metadata
}

fi['visible_to_llm'] = fi['feature'].map(lambda f: VISIBLE_IN_TEXT.get(f, False))
fi['visible_label'] = fi['visible_to_llm'].map({True: 'LLM-visible', False: 'Hidden from LLM'})

fig, ax = plt.subplots(figsize=(11, 6))
colors = fi['visible_to_llm'].map({True: '#4C72B0', False: '#C44E52'})
bars = ax.barh(fi['feature'], fi['importance'], color=colors, edgecolor='white', height=0.7)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='Visible in event text (LLM can access)'),
                   Patch(facecolor='#C44E52', label='Hidden from LLM (requires computation or absent)')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

ax.invert_yaxis()
ax.set_xlabel('XGBoost Feature Importance (gain)')
ax.set_title('XGBoost Top-20 Features: What GPT-4o Can and Cannot See', fontsize=12)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'xgboost_vs_llm_visibility.png', dpi=150, bbox_inches='tight')
plt.show()

n_visible = fi['visible_to_llm'].sum()
pct_importance_visible = fi.loc[fi['visible_to_llm'], 'importance'].sum() / fi['importance'].sum()
print(f'\nOf top-20 XGBoost features:')
print(f'  LLM-visible: {n_visible}/20 ({n_visible/20:.0%})')
print(f'  Importance covered by visible features: {pct_importance_visible:.1%}')
print(f'  Top feature (cam1_tear_ratio) is HIDDEN — requires ratio computation')
print(f'  Ranks 2-3 (grade_encoded, printer_encoded) are NOT in the event text at all')


Of top-20 XGBoost features:
  LLM-visible: 4/20 (20%)
  Importance covered by visible features: 16.0%
  Top feature (cam1_tear_ratio) is HIDDEN — requires ratio computation
  Ranks 2-3 (grade_encoded, printer_encoded) are NOT in the event text at all


C:\Users\SREE008\AppData\Local\Temp\ipykernel_34032\2535913915.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Failure Taxonomy

We annotate each v3 error with a failure type based on the reasoning text:

| failure_type | Description |
|---|---|
| `correct` | Prediction matches ground truth |
| `shortcut_visual_pattern` | Machine→Paper error; model cited visual sequence pattern (no_defect run + abrupt tear) rather than quantitative signal |
| `majority_class_bias` | Paper→Machine error; likely driven by class imbalance (84% machine in training) |
| `threshold_misexecution` | v5 specific: model cited injected threshold numbers but applied them to wrong class |

In [8]:
def assign_failure_type_v3(row):
    true_lbl = row['true_label']
    pred_lbl = row['predicted_label']
    reasoning = str(row.get('reasoning', '')).lower()

    if true_lbl == pred_lbl:
        return 'correct'

    if true_lbl == 'machine_problem' and pred_lbl == 'paper_problem':
        # Check for shortcut visual pattern in reasoning
        has_nodefect_tear = ('no_defect' in reasoning or 'no defect' in reasoning) and 'tear' in reasoning
        has_sudden = any(w in reasoning for w in ['sudden', 'abrupt', 'stable run'])
        if has_nodefect_tear or has_sudden:
            return 'shortcut_visual_pattern'
        return 'shortcut_other'

    if true_lbl == 'paper_problem' and pred_lbl == 'machine_problem':
        return 'majority_class_bias'

    return 'other'

df_v3['failure_type'] = df_v3.apply(assign_failure_type_v3, axis=1)

print('Failure type distribution (v3):')
print(df_v3['failure_type'].value_counts().to_string())
print()

# Show example shortcut reasoning
print('Example shortcut_visual_pattern reasoning:')
shortcuts = df_v3[df_v3['failure_type'] == 'shortcut_visual_pattern']
for _, row in shortcuts.head(4).iterrows():
    print(f'  [{row["event_id"]}] → {row["reasoning"][:120]}...')

Failure type distribution (v3):
failure_type
correct                    55
shortcut_other             16
shortcut_visual_pattern    15
other                      10
majority_class_bias         4

Example shortcut_visual_pattern reasoning:
  [FO59_20230309_171350] → The frequent 'tear' events interspersed with 'no_defect' suggest intrinsic paper weakness, aligning with paper problem e...
  [FO38_20240419_225059] → The event shows a long sequence of 'no_defect' followed by a large tear, which is indicative of a paper problem. The pre...
  [FO56_20230512_044343] → The event shows a long sequence of 'no_defect' followed by a large tear, which is indicative of a paper problem. The abr...
  [FO59_20220907_124728] → The event shows a long sequence of 'no_defect' followed by a large tear, which is indicative of a paper problem. The abr...


In [9]:
# Failure type pie chart
ft_counts = df_v3['failure_type'].value_counts()

FT_COLORS = {
    'correct': '#4C72B0',
    'shortcut_visual_pattern': '#DD8452',
    'shortcut_other': '#E07B54',
    'majority_class_bias': '#55A868',
    'other': '#AAAAAA'
}
pie_colors = [FT_COLORS.get(k, '#AAAAAA') for k in ft_counts.index]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    ft_counts.values, labels=ft_counts.index, colors=pie_colors,
    autopct='%1.0f%%', startangle=140, textprops={'fontsize': 11}
)
ax.set_title('v3 Failure Taxonomy (n=100 test events)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'failure_taxonomy_v3.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\SREE008\AppData\Local\Temp\ipykernel_34032\4149236201.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. The Architectural Failure: Why v5 Is Worse on Paper

We analyse how the injected XGBoost thresholds distort GPT-4o's classification of paper events.

The v5 prompt provides explicit rules like:  
- `grammage > 60 g/m² → paper signal`  
- `pap_len > 25000 m → paper signal`  

These are **correct** rules, but applying them requires reliable threshold comparison.  
GPT-4o can extract the numbers (58% of v3 reasonings already cite speed/grammage/paper_length),  
but its threshold comparisons are noisy — it sometimes classifies 57 g/m² as "strong paper signal"  
or misses a short roll of 135 m as a machine signal.

More importantly, the rules trained GPT-4o to **over-use** grammage and paper length  
while under-weighting `cam1_tear_ratio` (XGBoost's #1 feature), which requires computing  
a ratio from the sequence — an operation it cannot perform reliably.

In [10]:
# Reproduce v5 outcomes for the 10 events shown in notebook head()
# Source: LLMClassifier_Paper_Web_Breaks_v2.ipynb, Cell 11, df_results.head(10)

V5_SAMPLE = [
    {'event_id': 'FO45_20240205_122405', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'high',
     'reasoning': 'Grammage=38.0 g/m² (strong machine signal) + longest_no_defect_run covers full sequence → machine_problem'},
    {'event_id': 'FO45_20240123_031111', 'true_label': 'machine_problem',
     'predicted_label': 'paper_problem', 'confidence': 'medium',
     'reasoning': 'Grammage=60.0 g/m² (strong paper signal) + paper length=24453 m (neutral) → paper_problem'},
    {'event_id': 'FO37_20240820_050937', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'high',
     'reasoning': 'Pap_len=60.0 m (strong machine signal) + multiple rollenwechsel → machine_problem'},
    {'event_id': 'FO51_20240126_141512', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'medium',
     'reasoning': 'Grammage=48.0 g/m² (medium machine lean) + paper length=19302 m (neutral) → machine_problem'},
    {'event_id': 'FO59_20231109_162348', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'high',
     'reasoning': 'Pap_len=6.0 m (strong machine signal) + multiple rollenwechsel events → machine_problem'},
    {'event_id': 'FO59_20230309_171350', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'high',
     'reasoning': 'Paper length=226 m (strong machine signal) + grammage unknown → machine_problem'},
    {'event_id': 'FO59_20241205_081440', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'medium',
     'reasoning': 'Grammage=40 g/m² (slight machine lean) + no defects throughout sequence → machine_problem'},
    {'event_id': 'FO46_20240617_183745', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'medium',
     'reasoning': 'Grammage=57 g/m² (medium paper lean) + paper length=12683 m (neutral) + no defect pattern → machine_problem'},
    {'event_id': 'FO53_20240827_163949', 'true_label': 'machine_problem',
     'predicted_label': 'machine_problem', 'confidence': 'medium',
     'reasoning': 'Grammage=52 g/m² (medium paper lean) + paper length=7943 m (neutral) → machine_problem'},
    {'event_id': 'FO45_20240912_213856', 'true_label': 'machine_problem',
     'predicted_label': 'paper_problem', 'confidence': 'high',
     'reasoning': 'Grammage=100 g/m² (strong paper signal) + paper length=52763 m (strong paper signal) → paper_problem'},
]

df_v5_sample = pd.DataFrame(V5_SAMPLE)
df_v5_sample['prompt_condition'] = 'v5_xgboost_rules'
df_v5_sample['correct'] = df_v5_sample['true_label'] == df_v5_sample['predicted_label']

# Show what v5 cites vs what v3 cites for the same events
shared_ids = set(df_v5_sample['event_id']) & set(df_v3['event_id'])
print(f'Events appearing in both v3 and v5 samples: {len(shared_ids)}')

if shared_ids:
    compare = pd.merge(
        df_v3[df_v3['event_id'].isin(shared_ids)][['event_id','true_label','predicted_label','reasoning']],
        df_v5_sample[df_v5_sample['event_id'].isin(shared_ids)][['event_id','predicted_label','reasoning']],
        on='event_id', suffixes=('_v3','_v5')
    )
    for _, row in compare.head(3).iterrows():
        print(f"\nEvent: {row['event_id']} | True: {row['true_label']}")
        print(f"  v3 pred={row['predicted_label_v3']}  reasoning: {row['reasoning_v3'][:100]}")
        print(f"  v5 pred={row['predicted_label_v5']}  reasoning: {row['reasoning_v5'][:100]}")

Events appearing in both v3 and v5 samples: 10

Event: FO45_20240205_122405 | True: machine_problem
  v3 pred=machine_problem  reasoning: The event shows a long sequence of 'no_defect' without any defects or tears, indicating stable machi
  v5 pred=machine_problem  reasoning: Grammage=38.0 g/m² (strong machine signal) + longest_no_defect_run covers full sequence → machine_pr

Event: FO45_20240123_031111 | True: machine_problem
  v3 pred=machine_problem  reasoning: The presence of repeated 'Kantenfehler' without tears suggests a mechanical issue rather than a pape
  v5 pred=paper_problem  reasoning: Grammage=60.0 g/m² (strong paper signal) + paper length=24453 m (neutral) → paper_problem

Event: FO37_20240820_050937 | True: machine_problem
  v3 pred=machine_problem  reasoning: The frequent 'rollenwechsel' and periodic 'tear' occurrences suggest mechanical instability, consist
  v5 pred=machine_problem  reasoning: Pap_len=60.0 m (strong machine signal) + multiple rollenwechsel → machine_

## 8. Build Labeled Response Dataset

Creates the canonical labeled response CSV for this experiment.  
Columns: `event_id`, `true_label`, `predicted_label`, `prompt_condition`, `reasoning`, `failure_type`

**Note**: v5 full results (n=100) are not available as a CSV.  
The dataset below contains v3 full results (n=100) and v5 sample results (n=10 of 100).  
Re-running LLMClassifier_Paper_Web_Breaks_v2.ipynb will regenerate the complete v5 CSV.

In [11]:
def assign_failure_v5(row):
    if row['true_label'] == row['predicted_label']:
        reasoning = str(row.get('reasoning','')).lower()
        # Check if it cited thresholds correctly
        if any(w in reasoning for w in ['g/m²', 'g/m2', 'grammage', 'pap_len', 'paper length']):
            return 'correct_threshold_use'
        return 'correct'
    reasoning = str(row.get('reasoning','')).lower()
    if any(w in reasoning for w in ['g/m²', 'g/m2', 'grammage', 'strong paper signal', 'strong machine signal', 'pap_len']):
        return 'threshold_misexecution'
    return 'incorrect_other'

# Prepare v3 rows (all 100)
cols = ['event_id', 'true_label', 'predicted_label', 'prompt_condition', 'reasoning', 'failure_type']
df_v3_out = df_v3[['event_id', 'true_label', 'predicted_label', 'prompt_condition', 'reasoning', 'failure_type']].copy()

# Prepare v5 sample rows (10)
df_v5_sample['failure_type'] = df_v5_sample.apply(assign_failure_v5, axis=1)
df_v5_out = df_v5_sample[['event_id', 'true_label', 'predicted_label', 'prompt_condition', 'reasoning', 'failure_type']].copy()

# Concatenate
df_labeled = pd.concat([df_v3_out, df_v5_out], ignore_index=True)

print(f'Labeled response dataset: {len(df_labeled)} rows')
print(f'  v3 rows: {(df_labeled["prompt_condition"]=="v3_baseline_fewshot").sum()}')
print(f'  v5 rows: {(df_labeled["prompt_condition"]=="v5_xgboost_rules").sum()}')
print()
print('Failure type distribution:')
print(df_labeled['failure_type'].value_counts().to_string())

# Save
out_path = Path('../results/labeled_response_dataset.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_labeled.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'\nSaved: {out_path}')

df_labeled.head(10)

Labeled response dataset: 110 rows
  v3 rows: 100
  v5 rows: 10

Failure type distribution:
failure_type
correct                    55
shortcut_other             16
shortcut_visual_pattern    15
other                      10
correct_threshold_use       8
majority_class_bias         4
threshold_misexecution      2

Saved: ..\results\labeled_response_dataset.csv


,event_id,true_label,predicted_label,prompt_condition,reasoning,failure_type
0,FO45_20240205_122405,machine_problem,machine_problem,v3_baseline_fewshot,The event shows a long sequence of 'no_defect'...,correct
1,FO45_20240123_031111,machine_problem,machine_problem,v3_baseline_fewshot,The presence of repeated 'Kantenfehler' withou...,correct
2,FO37_20240820_050937,machine_problem,machine_problem,v3_baseline_fewshot,The frequent 'rollenwechsel' and periodic 'tea...,correct
3,FO51_20240126_141512,machine_problem,paper_problem,v3_baseline_fewshot,The sequence shows 'tear' events following 'de...,shortcut_other
4,FO59_20231109_162348,machine_problem,machine_problem,v3_baseline_fewshot,The abrupt 'tear' following 'rollenwechsel' an...,correct
5,FO59_20230309_171350,machine_problem,paper_problem,v3_baseline_fewshot,The frequent 'tear' events interspersed with '...,shortcut_visual_pattern
6,FO59_20241205_081440,machine_problem,machine_problem,v3_baseline_fewshot,The event shows a long sequence of 'no_defect'...,correct
7,FO46_20240617_183745,machine_problem,paper_problem,v3_baseline_fewshot,The presence of 'tear' events following 'Kante...,shortcut_other
8,FO53_20240827_163949,machine_problem,machine_problem,v3_baseline_fewshot,The sequence of 'defect' events without tears ...,correct
9,FO45_20240912_213856,machine_problem,machine_problem,v3_baseline_fewshot,The sequence shows 'rollenwechsel' followed by...,correct


## 9. Open Questions for SOAR I-1: SAE and Activation Analysis

This experiment establishes the behavioral baseline for the I-1 project.  
Three open questions motivate the next phase:

---

### 9.1 Can internal evidence distinguish genuine threshold-reasoning from shortcut-reasoning?

In v3, GPT-4o writes reasoning like *"long no_defect run followed by sudden tear → paper_problem"*.  
In v5, the **same model** writes *"Grammage=100 g/m² (strong paper signal) + paper length=52763 m → paper_problem"*.  
Surface behavior differs (different words, both can be wrong or right) but the question is:  
**Do internal activations differ between cases where the model genuinely compared a threshold vs  
cases where it pattern-matched the presence of a number in the prompt?**

Proposed SAE experiment:
- Run v3 and v5 prompts through an open-weight model (e.g., Llama-3 or Qwen)
- Extract residual stream activations at the token where the model commits to a class label
- Identify SAE features that fire differently on `shortcut_visual_pattern` vs `threshold_misexecution` errors
- Test whether a linear probe on those features predicts failure type above chance

---

### 9.2 What features activate on the injected threshold numbers?

The v5 prompt contains strings like `"grammage < 31 g/m² : machine"`.  
Does the model parse this as an executable rule, or as a semantic association?  
Hypothesis: SAE features for *comparison operations* (e.g., inequality evaluation features)  
should activate on numbers in the rule text if the model is "executing" the rule,  
but not if it is merely pattern-matching the label word associated with that range.

---

### 9.3 Does the collapse of paper recall in v5 have an activation signature?

Paper recall drops from 0.46 (v3) to 0.18 (v5). This is a **systematic** failure on the minority class.  
Possible mechanistic hypotheses:
- The v5 rules increase the prior for `machine_problem` in the model's feature space  
  (because most rule examples show machine boundaries)
- The injected thresholds activate a "apply the rule" circuit that overrides the paper examples
- The length/complexity of the v5 prompt causes the few-shot paper examples to be down-weighted
  in the attention pattern at classification time

Activation analysis can test these by comparing attention patterns to the few-shot paper examples  
vs the rule text across v3 and v5 conditions.

---

### 9.4 Can this serve as a dataset for matched I-1 problems?

The web break task can be adapted as a controlled I-1 problem set:  
- **Clean version**: Present event text with semantic few-shot examples only  
- **Subtly hinted**: Add qualitative domain guidance (e.g., "long clean runs suggest machine issues")  
- **Misleadingly hinted**: Inject plausible-but-wrong thresholds (e.g., reversed sign on cam1_tear_ratio)

The existing v3/v5 results provide the baseline behavioral dataset;  
constructing the subtly-hinted and intentionally-misleading variants is the natural I-1 extension.

In [12]:
print('Summary of artifacts produced by this notebook:')
print('  analysis/confusion_matrices_v3_v5.png')
print('  analysis/signal_mention_rates_v3.png')
print('  analysis/xgboost_vs_llm_visibility.png')
print('  analysis/failure_taxonomy_v3.png')
print('  results/labeled_response_dataset.csv')
print()
print('Key numbers for the README / application:')
print(f'  v3 paper recall: {m_v3["recall"]:.3f}  |  v5 paper recall: {m_v5["recall"]:.3f}')
print(f'  v3 paper F1:     {m_v3["f1"]:.3f}  |  v5 paper F1:     {m_v5["f1"]:.3f}')
print(f'  v3 accuracy:     {m_v3["accuracy"]:.3f}  |  v5 accuracy:     {m_v5["accuracy"]:.3f}')
print()
print('The recall collapse is the central finding:')
print(f'  Injecting correct quantitative rules reduced paper recall by '
      f'{(m_v5["recall"]-m_v3["recall"]):.0%} ({m_v3["recall"]:.2f} → {m_v5["recall"]:.2f}).')

Summary of artifacts produced by this notebook:
  analysis/confusion_matrices_v3_v5.png
  analysis/signal_mention_rates_v3.png
  analysis/xgboost_vs_llm_visibility.png
  analysis/failure_taxonomy_v3.png
  results/labeled_response_dataset.csv

Key numbers for the README / application:
  v3 paper recall: 0.364  |  v5 paper recall: 0.182
  v3 paper F1:     0.174  |  v5 paper F1:     0.121
  v3 accuracy:     0.620  |  v5 accuracy:     0.710

The recall collapse is the central finding:
  Injecting correct quantitative rules reduced paper recall by -18% (0.36 → 0.18).
